# Analyse des corrélations entre variables

Ce notebook regroupe les analyses de corrélation par grandes familles de variables afin d’identifier les variables fortement liées entre elles avant la modélisation.

**Méthodes utilisées**
- **Spearman** : quantitatif × quantitatif
- **V de Cramer** : qualitatif × qualitatif
- **Eta²** : quantitatif × qualitatif

Les résultats sont filtrés à partir de seuils définis par groupe.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd

from src.analyse_correlations import coef_cramer, epsilon_carre, coef_spearman, eta_carre, filtrer_coefficients, analyser_correlations

## 1. Chargement des données

In [ ]:
df = pd.read_excel("../data/base_adherents.xlsx")

print(f"Dimensions : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

## 2. Fonction d’analyse

La même procédure est appliquée à chaque groupe de variables. Cela évite de répéter les mêmes cellules pour chaque analyse.

In [ ]:
def analyser_correlations(df, variables, seuil_spearman=0.7, seuil_cramer=0.7, seuil_eta=0.5, batch_size=14):
    """Calcule et filtre les trois types de corrélations pour un groupe de variables."""
    data = df[variables].copy()

    resultats = {
        "spearman": filtrer_coefficients(
            coef_spearman(data, batch_size=batch_size),
            seuil=seuil_spearman,
            est_symetrique=True,
        ),
        "cramer": filtrer_coefficients(
            coef_cramer(data, batch_size=batch_size),
            seuil=seuil_cramer,
            est_symetrique=True,
        ),
        "eta_carre": filtrer_coefficients(
            eta_carre(data, batch_size=10),
            seuil=seuil_eta,
            est_symetrique=False,
        ),
    }

    return resultats

## 3. Définition des groupes de variables

Les variables sont regroupées selon leur rôle dans l’analyse :
1. descriptives de l’organisation et des adhérents ;
2. historique des adhésions et des contacts ;
3. caractéristiques des adhérents ;
4. communications ;
5. événements.

In [ ]:
groupes = {
    "descriptives": {
        "variables": [
            "a_deja_demissionne",
            "Organisation - Marketing Budget 2019",
            "has_F_level",
            "has_M_level",
            "nb_contacts",
            "department_RH",
            "department_Marketing",
            "department_Direction_générale",
            "department_Juridique",
            "department_Médias",
            "department_Impact",
            "department_Insights",
            "department_Affaires_Publiques",
            "department_Performance_digitale",
            "has_VIP",
            "has_active_speaker",
            "average_anticipation_days_by_group",
            "target",
        ],
        "seuil_eta": 0.5,
    },

    "adhesions_et_contacts": {
        "variables": [
            "GROUPE - Année adhésion *",
            "GROUPE - Année démission *",
            "GROUPE - Nombre d'adhésion",
            "duree_derniere_adhesion",
            "a_deja_demissionne",
            "Organisation - Membership status",
            "days_since_youngest_contact",
            "youngest_contact_year",
            "creation_date_oldest_contact",
            "creation_date_youngest_contact",
            "oldest_contact_year",
            "days_since_oldest_contact",
            "creation_date_oldest_current_member",
            "creation_date_youngest_current_member",
            "oldest_current_member_year",
            "youngest_current_member_year",
            "days_since_oldest_current_member",
            "days_since_youngest_current_member",
            "days_since_last_online",
            "nb_contacts",
            "nb_contacts_ayant_recu_mail",
            "nb_current_contacts",
            "target",
        ],
        "seuil_eta": 0.5,
    },

    "adherents": {
        "variables": [
            "has_F_level",
            "has_M_level",
            "has_C_level",
            "has_VIP",
            "has_active_speaker",
            "Organisation - Marketing Budget 2019",
            "Organisation - Sector of Activities",
            "department_RH",
            "department_Marketing",
            "department_Direction_générale",
            "department_Juridique",
            "department_Médias",
            "department_Impact",
            "department_Insights",
            "department_Affaires_Publiques",
            "department_Performance_digitale",
            "nb_contacts",
            "target",
        ],
        "seuil_eta": 0.5,
    },

    "communications": {
        "variables": [
            "recipient_status_pct_Active, relation receives e-mail",
            "recipient_status_pct_Inactive, relation will not receive mailings (Reason: Manually disabled)",
            "recipient_status_pct_Inactive, relation will not receive mailings (Reason: Too many bounces)",
            "comm_accept_pct_Communication - Communautés",
            "comm_accept_pct_Communication - Newsletters",
            "comm_accept_pct_Communication - Partner communications",
            "comm_accept_pct_Communication - Veille juridique",
            "nb_moyen_emails_recus_par_contact_par_an",
            "avg_taux_ouverture",
            "avg_taux_click",
            "avg_taux_click_sur_overture",
            "avg_days_since_last_open",
            "days_since_most_recent_open",
            "recipient_status_nb_Active, relation receives e-mail",
            "recipient_status_nb_Inactive, relation will not receive mailings (Reason: Manually disabled)",
            "recipient_status_nb_Inactive, relation will not receive mailings (Reason: Too many bounces)",
            "Organisation - Communication - Communautés",
            "comm_accept_nb_Communication - Communautés",
            "comm_accept_nb_Communication - Newsletters",
            "comm_accept_nb_Communication - Partner communications",
            "comm_accept_nb_Communication - Veille juridique",
            "nb_unsubscribed_by_relation",
            "unsub_rate_pct",
            "avg_clicks_per_contact",
            "avg_mails_received_per_contact",
            "avg_mails_open_per_contact",
            "avg_mails_clicked_per_contact",
            "avg_days_open_per_contact",
            "avg_open_delay_hours_by_contact",
            "avg_days_since_last_click",
            "days_since_most_recent_click",
            "target",
        ],
        "seuil_eta": 0.5,
    },

    "evenements": {
        "variables": [
            "avg_days_since_last_registration",
            "nb_moyen_inscriptions_par_contact_par_an",
            "avg_presence_rate_by_relation",
            "avg_invitation_reactivity_by_relation",
            "average_anticipation_days_by_group",
            "avg_days_since_last_participation",
            "avg_present_count_per_contact",
            "registered_count_by_organisation",
            "nb_inscriptions_par_organisation_par_an",
            "avg_registered_count_per_contact",
            "days_since_most_recent_participation",
            "days_since_most_recent_registration",
            "avg_spontaneous_participation_rate_by_relation",
            "avg_cancelled_count_by_relation",
            "avg_opted_out_count_by_relation",
            "avg_reserve_list_count_by_relation",
            "avg_no_reaction_count_by_relation",
            "nb_contacts",
            "target",
        ],
        "seuil_eta": 0.5,
    },
}

## 4. Analyse des groupes

Les résultats sont stockés dans un dictionnaire `resultats` pour pouvoir être consultés sans recréer les calculs.

In [ ]:
resultats = {}

for nom, config in groupes.items():
    resultats[nom] = analyser_correlations(
        df,
        config["variables"],
        seuil_spearman=0.7,
        seuil_cramer=0.7,
        seuil_eta=config["seuil_eta"],
    )

print("Analyses terminées :", ", ".join(resultats))

### Consulter les résultats

Exemple pour le groupe `communications`. Remplacer le nom du groupe si besoin.

In [ ]:
resultats["communications"]["spearman"]

In [ ]:
resultats["communications"]["cramer"]

In [ ]:
resultats["communications"]["eta_carre"]